# Lab | Web Scraping

Welcome to the "Books to Scrape" Web Scraping Adventure Lab!

**Objective**

In this lab, we will embark on a mission to unearth valuable insights from the data available on Books to Scrape, an online platform showcasing a wide variety of books. As data analyst, you have been tasked with scraping a specific subset of book data from Books to Scrape to assist publishing companies in understanding the landscape of highly-rated books across different genres. Your insights will help shape future book marketing strategies and publishing decisions.

**Background**

In a world where data has become the new currency, businesses are leveraging big data to make informed decisions that drive success and profitability. The publishing industry, much like others, utilizes data analytics to understand market trends, reader preferences, and the performance of books based on factors such as genre, author, and ratings. Books to Scrape serves as a rich source of such data, offering detailed information about a diverse range of books, making it an ideal platform for extracting insights to aid in informed decision-making within the literary world.

**Task**

Your task is to create a Python script using BeautifulSoup and pandas to scrape Books to Scrape book data, focusing on book ratings and genres. The script should be able to filter books with ratings above a certain threshold and in specific genres. Additionally, the script should structure the scraped data in a tabular format using pandas for further analysis.

**Expected Outcome**

A function named `scrape_books` that takes two parameters: `min_rating` and `max_price`. The function should scrape book data from the "Books to Scrape" website and return a `pandas` DataFrame with the following columns:

**Expected Outcome**

- A function named `scrape_books` that takes two parameters: `min_rating` and `max_price`.
- The function should return a DataFrame with the following columns:
  - **UPC**: The Universal Product Code (UPC) of the book.
  - **Title**: The title of the book.
  - **Price (£)**: The price of the book in pounds.
  - **Rating**: The rating of the book (1-5 stars).
  - **Genre**: The genre of the book.
  - **Availability**: Whether the book is in stock or not.
  - **Description**: A brief description or product description of the book (if available).
  
You will execute this script to scrape data for books with a minimum rating of `4.0 and above` and a maximum price of `£20`. 

Remember to experiment with different ratings and prices to ensure your code is versatile and can handle various searches effectively!

**Resources**

- [Beautiful Soup Documentation](https://www.crummy.com/software/BeautifulSoup/bs4/doc/)
- [Pandas Documentation](https://pandas.pydata.org/pandas-docs/stable/index.html)
- [Books to Scrape](https://books.toscrape.com/)


**Hint**

Your first mission is to familiarize yourself with the **Books to Scrape** website. Navigate to [Books to Scrape](http://books.toscrape.com/) and explore the available books to understand their layout and structure. 

Next, think about how you can set parameters for your data extraction:

- **Minimum Rating**: Focus on books with a rating of 4.0 and above.
- **Maximum Price**: Filter for books priced up to £20.

After reviewing the site, you can construct a plan for scraping relevant data. Pay attention to the details displayed for each book, including the title, price, rating, and availability. This will help you identify the correct HTML elements to target with your scraping script.

Make sure to build your scraping URL and logic based on the patterns you observe in the HTML structure of the book listings!


---

**Best of luck! Immerse yourself in the world of books, and may the data be with you!**

**Important Note**:

In the fast-changing online world, websites often update and change their structures. When you try this lab, the **Books to Scrape** website might differ from what you expect.

If you encounter issues due to these changes, like new rules or obstacles preventing data extraction, don’t worry! Get creative.

You can choose another website that interests you and is suitable for scraping data. Options like Wikipedia, The New York Times, or even library databases are great alternatives. The main goal remains the same: extract useful data and enhance your web scraping skills while exploring a source of information you enjoy. This is your opportunity to practice and adapt to different web environments!

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

In [2]:
url = "https://books.toscrape.com/catalogue/page-1.html"
response = requests.get(url)
print(response.status_code)   # 200 means it worked

soup = BeautifulSoup(response.content, "html.parser")
books = soup.find_all("article", class_="product_pod")
print(len(books))             # 20 books per page

first_book = books[0]
print(first_book.h3.a["title"])
print(first_book.find("p", class_="price_color").text)
print(first_book.find("p", class_="star-rating")["class"])

200
20
A Light in the Attic
£51.77
['star-rating', 'Three']


In [3]:
def scrape_books(min_rating, max_price):
    base_url = "https://books.toscrape.com/catalogue/"
    rating_words = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}

    books_data = []

    for page in range(1, 51):
        page_url = f"{base_url}page-{page}.html"
        response = requests.get(page_url)
        soup = BeautifulSoup(response.content, "html.parser")

        for book in soup.find_all("article", class_="product_pod"):
            # Rating from the list page: "Four" → 4
            rating_word = book.find("p", class_="star-rating")["class"][1]
            rating = rating_words[rating_word]

            # Price from the list page: "£17.93" → 17.93
            price_text = book.find("p", class_="price_color").text
            price = float(price_text.replace("£", "").replace("Â", ""))

            # Skip books that don't pass the filter
            if rating < min_rating or price > max_price:
                continue

            # Open the book's own page for the other details
            book_url = base_url + book.h3.a["href"]
            book_response = requests.get(book_url)
            book_soup = BeautifulSoup(book_response.content, "html.parser")

            # Title
            title = book_soup.find("h1").text

            # UPC and availability from the table
            table = book_soup.find("table", class_="table table-striped")
            table_data = {}
            for row in table.find_all("tr"):
                table_data[row.th.text] = row.td.text
            upc = table_data["UPC"]
            availability = table_data["Availability"]

            # Genre from the breadcrumb: Home > Books > Genre > Title
            genre = book_soup.find("ul", class_="breadcrumb").find_all("a")[2].text

            # Description (some books don't have one)
            description_div = book_soup.find("div", id="product_description")
            if description_div:
                description = description_div.find_next_sibling("p").text
            else:
                description = ""

            books_data.append({
                "UPC": upc,
                "Title": title,
                "Price (£)": price,
                "Rating": rating,
                "Genre": genre,
                "Availability": availability,
                "Description": description
            })

    return pd.DataFrame(books_data)

In [4]:
result = scrape_books(min_rating=4, max_price=20)

print(result.shape)
result.head()

(75, 7)


,UPC,Title,Price (£),Rating,Genre,Availability,Description
0,ce6396b0f23f6ecc,Set Me Free,17.46,5,Young Adult,In stock (19 available),Aaron Ledbetter’s future had been planned out ...
1,6258a1f6a6dcfe50,The Four Agreements: A Practical Guide to Pers...,17.66,5,Spirituality,In stock (18 available),"In The Four Agreements, don Miguel Ruiz reveal..."
2,6be3beb0793a53e7,Sophie's World,15.94,5,Philosophy,In stock (18 available),A page-turning novel that is also an explorati...
3,657fe5ead67a7767,Untitled Collection: Sabbath Poems 2014,14.27,4,Poetry,In stock (16 available),"More than thirty-five years ago, when the weat..."
4,51653ef291ab7ddc,This One Summer,19.49,4,Sequential Art,In stock (16 available),"Every summer, Rose goes with her mom and dad t..."


In [5]:
cheap_top = scrape_books(min_rating=5, max_price=15)
print(cheap_top.shape)

(23, 7)


In [6]:
# which genres have the most highly-rated cheap boooks
print(result["Genre"].value_counts().head(10))
print(result.groupby("Genre")["Price (£)"].mean().round(2).sort_values())

# how many 4-star vs 5-star
print(result["Rating"].value_counts())

Genre
Default           10
Sequential Art     9
Young Adult        7
Nonfiction         6
Fiction            5
Poetry             4
Food and Drink     4
Fantasy            3
Romance            3
Philosophy         2
Name: count, dtype: int64
Genre
Art                   10.29
Science               11.52
Childrens             11.88
Food and Drink        12.20
Mystery               12.29
Business              12.61
Horror                12.75
Autobiography         12.91
Fiction               13.16
Humor                 13.29
Young Adult           13.74
Fantasy               14.24
Nonfiction            14.70
Default               14.88
Classics              15.08
Sequential Art        15.24
Poetry                15.34
Adult Fiction         15.36
Romance               15.65
Womens Fiction        16.45
Philosophy            16.69
Historical Fiction    16.97
Add a comment         17.49
Spirituality          17.66
Christian Fiction     17.97
Science Fiction       19.07
Erotica               19